## Validation Script for Porting the Nugraph DA code 

#### Set autoreloading
This extension will automatically update with any changes to packages in real time

In [1]:
%load_ext autoreload
%autoreload 2

#### Append the path of the Nugraph Base conda libraries

In [2]:
import sys
user = "twalton"
sys.path.append('/home/%s/.conda/envs/NugraphBase/lib/python3.10/site-packages' % user)

#### Import packages for training

In [3]:
from pathlib import Path
import nugraph as ng
import pytorch_lightning as pl
print(ng.__file__)

/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages/torch/cuda/__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/nugraph/nugraph/__init__.py


#### Determine the run configuration 

In [4]:
"""
Define the source and target files
Define the dataset names
"""

"""
# small datasets, 5.3 GB and 1.6 GB
source_filename="/scratch/7DayLifetime/cerati/concat-final-makeup.tiny.gnn.h5"
target_filename="/scratch/7DayLifetime/cerati/icarus_numi_2d.gnn.h5"
dataset_name = "MicroBoone-Icarus-Small" 
"""

# large datasets, 53 and 7 GB
source_filename="/scratch/7DayLifetime/cerati/concat-final-makeup.half.gnn.h5"
target_filename="/scratch/7DayLifetime/cerati/icarus-numi-1d.gnn.h5"
dataset_name = "No-DA-All-Decoders-MicroBoone-Icarus-Large" 

input_files = 2
trainer_epochs = 35
run_domain_adaptation = False

# tuned the data batch size according to how you are running the decoders
fnc_name = "dann" #dann, mmd, semantic, sinkhorn 
data_batch_size = 64*2

run_event_decoder = True
run_semantic_decoder = True
run_filter_decoder = True

model_da_loss_fnc_name = None if input_files == 1 else fnc_name
print( "model_da_loss_fnc_name [", model_da_loss_fnc_name.upper(), "]" )

if run_domain_adaptation:
   model_warmup_epochs = 0 if input_files == 1 else 10
else:
   model_warmup_epochs = trainer_epochs

subname = fnc_name.upper() if run_domain_adaptation else "OFF"

if input_files == 1:
   os.environ["NUGRAPH_LOG"]='/home/%s/NuGraphLogs/NuGraph/EPOCH%d-WARMUP%d' % (user,trainer_epochs,model_warmup_epochs)
else:
   os.environ["NUGRAPH_LOG"]='/home/%s/NuGraphLogs/NuGraph-DA-%s/EPOCH%d-WARMUP%d' % (user,subname,trainer_epochs,model_warmup_epochs)

print( "Files are in the directory [", os.environ["NUGRAPH_LOG"], "]" )

model_da_loss_fnc_name [ DANN ]
Files are in the directory [ /home/twalton/NuGraphLogs/NuGraph-DA-OFF/EPOCH35-WARMUP35 ]


#### Set the data and module to use

In [5]:
Data  = ng.data.NuGraphDataModule
Model = ng.models.NuGraph3

#### Declare and configure the data module

In [6]:
if input_files == 1:
   nudata = Data(model=Model, batch_size=data_batch_size, data_source_path=source_filename)
elif input_files == 2:
   nudata = Data(model=Model, batch_size=data_batch_size, data_source_path=source_filename, data_target_path=target_filename)

nudata.event_classes = ['cc_nue', 'cc_numu', 'cc_nutau', 'nc']
print(nudata)
print(nudata.semantic_classes)
print(nudata.event_classes)

{Train dataloader: size=29953}
{Validation dataloader: size=1663}
{Test dataloader: size=1663}
{Predict dataloader: None}
['MIP', 'HIP', 'shower', 'michel', 'diffuse']
['cc_nue', 'cc_numu', 'cc_nutau', 'nc']


#### Configure network

In [7]:
nugraph = Model(
    in_features=5,  
    hit_features=128,
    nexus_features=32,
    instance_features=32,
    interaction_features=32,
    semantic_classes=nudata.semantic_classes, 
    event_classes=nudata.event_classes,
    num_iters=5,
    event_head=run_event_decoder,
    semantic_head=run_semantic_decoder,
    filter_head=run_filter_decoder,
    vertex_head=False,
    instance_head=False,
    use_checkpointing=True,
    lr=0.001,
    da_loss_fnc_name=model_da_loss_fnc_name,
    warmup_epochs=model_warmup_epochs)

#### Configure logger and callbacks
Declare a TensorBoard logger and define the output directory, so we can monitor network training. Also, define a callback so we can monitor learning rate evolution.

In [8]:
from pytorch_lightning.loggers import TensorBoardLogger
from datetime import datetime
now = datetime.now()
folder_name = "%s-%s" % (dataset_name, now.strftime("%Y-%m-%d-%H"))
print(folder_name)

No-DA-All-Decoders-MicroBoone-Icarus-Large-2026-09-03-19


In [9]:
logdir = Path(os.environ["NUGRAPH_LOG"])
logdir.mkdir(parents=True, exist_ok=True)
logger = TensorBoardLogger(save_dir=logdir,name=folder_name) 
callbacks = [
    pl.callbacks.LearningRateMonitor(logging_interval="step"),
    pl.callbacks.ModelCheckpoint(monitor="loss/val", mode="min"),
]

#### Declare trainer and run training
First, we set the training device. To train with a GPU, pass an integer; otherwise, it defaults to CPU training. We then instantiate a PyTorch Lightning trainer and run the training stage, iterating over all batches in the training and validation datasets to optimize model parameters, logging metrics to TensorBoard.

In [ ]:
device_number = 0
accelerator, devices = ng.util.configure_device(device_number)
print(accelerator)
print(devices)

trainer = pl.Trainer(accelerator=accelerator,
                     devices=devices,
                     max_epochs=trainer_epochs,
                     logger=logger,
                     callbacks=callbacks)
trainer.fit(nugraph, datamodule=nudata)
trainer.test(datamodule=nudata)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


gpu
[0]


You are using a CUDA device ('NVIDIA A100 80GB PCIe MIG 2g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type            | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | encoder          | Encoder         | 988    | train | 0    
1 | core_net         | NuGraphCore     | 141 K  | train | 0    
2 | event_decoder    | EventDecoder    | 201    | train | 0    
3 | semantic_decoder | SemanticDecoder | 1.2 K  | train | 0    
4 | filter_decoder   | FilterDecoder   | 8.5 K  | train | 0    
---------------------------------------------------------------------
152 K     Trainable params
22        Non-trainable params
1

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'y_vtx'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):


Validation epoch ended!


/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

[Epoch 0] and the DA is OFF for event_decoder (warmup phase)
[Epoch 0] and the DA is OFF for semantic_decoder (warmup phase)
[Epoch 0] and the DA is OFF for filter_decoder (warmup phase)


/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'y_vtx'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
